In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")

In [10]:
def normalize_day_type(
    df,
    day_type_col,
    day_of_week_col
):
    """
    Reclassifies 'Holiday' rows into Weekday/Saturday/Sunday
    based on the day of week.
    """
    return df.apply(
        lambda row: (
            'Weekday'
            if row[day_type_col] == 'Holiday'
            and row[day_of_week_col] in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
            else (
                'Saturday'
                if row[day_type_col] == 'Holiday'
                and row[day_of_week_col] == 'Saturday'
                else (
                    'Sunday'
                    if row[day_type_col] == 'Holiday'
                    and row[day_of_week_col] == 'Sunday'
                    else row[day_type_col]
                )
            )
        ),
        axis=1
    )

In [11]:
# Function to count number of days of a Day_Type in a service period
def count_day_type_days(start, end, day_type):
    dates = pd.date_range(start, end)
    day_type_upper = day_type.upper()
    if day_type_upper == 'WEEKDAY':
        return sum(d.weekday() < 5 for d in dates)  
    elif day_type_upper == 'SATURDAY':
        return sum(d.weekday() == 5 for d in dates)
    elif day_type_upper == 'SUNDAY':
        return sum(d.weekday() == 6 for d in dates)
    else:
        return 0

In [12]:
def add_day_type(df, date_col, output_col='Day Type'):
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [13]:
bart_ridership['Day Type'] = normalize_day_type(
    bart_ridership,
    day_type_col='Day Type',
    day_of_week_col='Day of Week'
)

bart_ridership['exposure'] = bart_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

bart_ridership.rename(columns={
    'Station': 'Stop',
    'Station Name': 'Stop Name',
    'Entries': 'Boardings'
}, inplace=True)

In [14]:
# Compute exposure: number of days of that Day_Type in the period
big_blue_bus_ridership['exposure'] = big_blue_bus_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['SERVICE_DAY']),
    axis=1
)

# Compute total boardings for the period
big_blue_bus_ridership['Boardings'] = big_blue_bus_ridership['AVERAGE_DAILY_BOARDINGS'] * big_blue_bus_ridership['exposure']

# Rename columns
big_blue_bus_ridership.rename(columns={
    'SERVICE_DAY': 'Day_Type',
    'STOP_ID': 'Stop',
    'STOP_NAME': 'Stop Name'
}, inplace=True)


For Caltrain Ridership, dropping "Holiday" data since we just have average ridership for the month. 

In [15]:
caltrain_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2520 entries, 0 to 2519
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Month               2520 non-null   object        
 1   Origin Station      2520 non-null   object        
 2   Caltrain Ridership  2520 non-null   float64       
 3   Date Type           2520 non-null   object        
 4   Average Ridership   2520 non-null   float64       
 5   start_date          2520 non-null   datetime64[ns]
 6   end_date            2520 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), object(3)
memory usage: 137.9+ KB


In [16]:
caltrain_ridership = caltrain_ridership[caltrain_ridership['Date Type'] != 'Holiday']


# Compute exposure: number of days of that Day_Type in the period
caltrain_ridership['exposure'] = caltrain_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Date Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
caltrain_ridership['Boardings'] = caltrain_ridership['Average Ridership'] * caltrain_ridership['exposure']

In [17]:
caltrain_ridership.sample(5)

,Month,Origin Station,Caltrain Ridership,Date Type,Average Ridership,start_date,end_date,exposure,Boardings
559,January 2024,San Antonio,8712.921942,Weekday,343.602824,2024-01-01,2024-01-31,23,7902.864949
8,July 2025,College Park,202.058311,Weekday,9.002853,2025-07-01,2025-07-31,23,207.065609
1795,February 2024,San Mateo,18649.189683,Sunday,240.756976,2024-02-01,2024-02-29,4,963.027903
439,May 2024,San Antonio,11409.349902,Weekday,428.977657,2024-05-01,2024-05-31,23,9866.486107
631,July 2025,Bayshore,6839.629852,Saturday,170.317530,2025-07-01,2025-07-31,4,681.270118


In [18]:
culver_citybus_ridership.sample(5)

,Day Of Week,Route,Direction,Stop ID,Stop Name,AVG On,AVG Off,start_date,end_date,route_id,day_type
1035,3-Sunday,3-Crosstown,outbound,354,Westwood Blvd/Pico Blvd,9.7,3.1,2025-07-14,2025-08-25,3,Sunday
1065,3-Sunday,3-Crosstown,outbound,439,Jefferson Blvd/Cota St,2.8,12.4,2025-07-14,2025-08-25,3,Sunday
313,1-Weekday,4-Jefferson Boulevard,Inbound,424,ObamaBlvd/KalsmanDr,0.2,2.2,2025-07-14,2025-08-25,4,Weekday
980,3-Sunday,3-Crosstown,Inbound,302,Green Valley Cir/Sepulveda Blvd,0.5,0.5,2025-07-14,2025-08-25,3,Sunday
863,2-Saturday,6-Sepulveda Boulevard,outbound,696,WESTCHESTER/AIRPORT,0.0,0.0,2025-07-14,2025-08-25,6,Saturday


In [19]:
group_cols = ['Day Of Week', 'Route', 'Stop ID', 'Stop Name', 
              'route_id', 'start_date', 'end_date', 'day_type']

# Sum AVG On and AVG Off
total_ridership_culver = culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg({
    'AVG On': 'sum'
})

Culver city : Missing records for either inbound or outbound for certain stops. For now, we are considering no service/no stop for that direction.

In [20]:
total_ridership_culver.head(5)

,Day Of Week,Route,Stop ID,Stop Name,route_id,start_date,end_date,day_type,AVG On
0,1-Weekday,1-Washington Boulevard,101,WindwardAve/MainSt,1,2025-07-14,2025-08-25,Weekday,111.2
1,1-Weekday,1-Washington Boulevard,102,Pacific Ave/N Venice Blvd,1,2025-07-14,2025-08-25,Weekday,31.7
2,1-Weekday,1-Washington Boulevard,103,Washington Blvd/Pacific Ave,1,2025-07-14,2025-08-25,Weekday,84.2
3,1-Weekday,1-Washington Boulevard,104,Washington Blvd/Via Dolce,1,2025-07-14,2025-08-25,Weekday,39.4
4,1-Weekday,1-Washington Boulevard,105,Washington Blvd/Via Marina,1,2025-07-14,2025-08-25,Weekday,42.4


In [21]:
# Compute exposure: number of days of that Day_Type in the period
total_ridership_culver['exposure'] = total_ridership_culver.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['day_type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_ridership_culver['Boardings'] = total_ridership_culver['AVG On'] * total_ridership_culver['exposure']

In [22]:
foothill_transit_ridership['day_type'].unique()

array(['weekday', 'holiday', 'weekend'], dtype=object)

In [23]:
foothill_transit_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694858 entries, 0 to 694857
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              694858 non-null  datetime64[ns]
 1   route_short_name  694858 non-null  int64         
 2   direction         694858 non-null  object        
 3   stop_code         694858 non-null  int64         
 4   stop_lat          694858 non-null  float64       
 5   stop_lon          694858 non-null  float64       
 6   boardings         694858 non-null  int64         
 7   alightings        694858 non-null  int64         
 8   start_date        694858 non-null  datetime64[ns]
 9   end_date          694858 non-null  datetime64[ns]
 10  day_type          694858 non-null  object        
dtypes: datetime64[ns](3), float64(2), int64(4), object(2)
memory usage: 58.3+ MB


In [24]:
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col='date')

In [25]:
group_cols = ['date', 'route_short_name', 'stop_code', 'stop_lat', 'stop_lon', 
              'start_date', 'end_date', 'day_type', 'Day Type']

# Sum AVG On and AVG Off
total_ridership_foothill = foothill_transit_ridership.groupby(
    group_cols, as_index=False).agg({
    'boardings': 'sum'
})

In [26]:
# Compute exposure: number of days of that Day_Type in the period
total_ridership_foothill['exposure'] = total_ridership_foothill.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_ridership_foothill['Boardings'] = total_ridership_foothill['boardings'] * total_ridership_foothill['exposure']

In [27]:
fresno_area_express_ridership.head(5)

,Date,StopID,StopLabel,ProjectedBoarding,ProjectedAlighting,start_date,end_date,day_type
0,2024-09-01,5,NE BRAWLEY - SHIELDS,44.691729,29.748092,2024-09-01,2024-09-01,weekend
1,2024-09-01,6,SE SHAW - BRAWLEY,7.000000,0.000000,2024-09-01,2024-09-01,weekend
2,2024-09-01,7,SW SHAW - WEST,20.000000,20.000000,2024-09-01,2024-09-01,weekend
3,2024-09-01,8,SE SHAW - BLACKSTONE,79.000000,17.000000,2024-09-01,2024-09-01,weekend
4,2024-09-01,9,SE SHAW - FIRST,63.000000,29.000000,2024-09-01,2024-09-01,weekend


bart_ridership, caltrain, fresno_area_express_ridership no route information

In [28]:
fresno_area_express_ridership['day_type'].unique()

array(['weekend', 'holiday', 'weekday'], dtype=object)

In [29]:
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col='Date')

# Compute exposure: number of days of that Day_Type in the period
fresno_area_express_ridership['exposure'] = fresno_area_express_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
fresno_area_express_ridership['Boardings'] = fresno_area_express_ridership['ProjectedBoarding'] * fresno_area_express_ridership['exposure']

In [30]:
fresno_area_express_ridership.head(5)

,Date,StopID,StopLabel,ProjectedBoarding,ProjectedAlighting,start_date,end_date,day_type,Day Type,exposure,Boardings
0,2024-09-01,5,NE BRAWLEY - SHIELDS,44.691729,29.748092,2024-09-01,2024-09-01,weekend,Sunday,1,44.691729
1,2024-09-01,6,SE SHAW - BRAWLEY,7.000000,0.000000,2024-09-01,2024-09-01,weekend,Sunday,1,7.000000
2,2024-09-01,7,SW SHAW - WEST,20.000000,20.000000,2024-09-01,2024-09-01,weekend,Sunday,1,20.000000
3,2024-09-01,8,SE SHAW - BLACKSTONE,79.000000,17.000000,2024-09-01,2024-09-01,weekend,Sunday,1,79.000000
4,2024-09-01,9,SE SHAW - FIRST,63.000000,29.000000,2024-09-01,2024-09-01,weekend,Sunday,1,63.000000


In [31]:
gold_coast_transit_ridership.head(5)

,day_of_week,route,direction,stop_id,unknown,stop_name,total_on,total_off,total_activity,cumulative_load,lat,lon,start_date,end_date
0,Weekday,1A,North,4THBST2,19,4th & B St,2,61,62,97,34.198975,-119.179621,2025-05-01,2025-05-31
1,Weekday,1A,South,4THBST1,1,4th & B St,70,2,72,188,34.199066,-119.179574,2025-05-01,2025-05-31
2,Weekday,1B,North,4THBST2,21,4th & B St,1,56,57,115,34.198975,-119.179621,2025-05-01,2025-05-31
3,Weekday,1B,South,4THBST1,1,4th & B St,63,2,65,183,34.199066,-119.179574,2025-05-01,2025-05-31
4,Weekday,1B,South,BAR5TH1,16,Bard & 5th,2,6,7,132,34.161169,-119.194368,2025-05-01,2025-05-31


In [32]:
gold_coast_transit_ridership['direction'].unique()

array(['North', 'South', 'Loop', 'East', 'West', 'PM', 'AM', 'A'],
      dtype=object)

Skipping this for now.
- Stop id is custom generated and cannot be used.
- Some rows don't have route numbers
- Will add this agency later

In [33]:
gold_coast_transit_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1046 entries, 0 to 1045
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   day_of_week      1046 non-null   object        
 1   route            1032 non-null   object        
 2   direction        1046 non-null   object        
 3   stop_id          1046 non-null   object        
 4   unknown          1046 non-null   int64         
 5   stop_name        1046 non-null   object        
 6   total_on         1046 non-null   int64         
 7   total_off        1046 non-null   int64         
 8   total_activity   1046 non-null   int64         
 9   cumulative_load  1046 non-null   int64         
 10  lat              1046 non-null   float64       
 11  lon              1046 non-null   float64       
 12  start_date       1046 non-null   datetime64[ns]
 13  end_date         1046 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(

In [40]:
golden_gate_park_shuttle_ridership['Day Type'].unique()

array(['Weekday', 'Saturday', 'Sunday'], dtype=object)

In [42]:
golden_gate_park_shuttle_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6570 entries, 0 to 6569
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        6570 non-null   datetime64[ns]
 1   Month       6570 non-null   int64         
 2   Day Type    6570 non-null   object        
 3   Day         6570 non-null   object        
 4   Stop        6570 non-null   object        
 5   Ridership   6570 non-null   int64         
 6   start_date  6570 non-null   datetime64[ns]
 7   end_date    6570 non-null   datetime64[ns]
 8   direction   4380 non-null   object        
dtypes: datetime64[ns](3), int64(2), object(4)
memory usage: 462.1+ KB


In [38]:
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col='Date')

In [53]:
golden_gate_park_shuttle_ridership["Stop"] = golden_gate_park_shuttle_ridership["Stop"].str.replace(
    r"\s*-?\s*(EB|WB|NB|SB)$", 
    "", 
    regex=True
)

group_cols = [
    "Date", "Month", "Day Type", "Day", "Stop", "start_date", "end_date"]

# Sum AVG On and AVG Off
total_golden_gate_park_shuttle_ridership = golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg({
    'Ridership': 'sum'
})

In [56]:
# Compute exposure: number of days of that Day_Type in the period
total_golden_gate_park_shuttle_ridership['exposure'] = total_golden_gate_park_shuttle_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_golden_gate_park_shuttle_ridership['Boardings'] = total_golden_gate_park_shuttle_ridership['Ridership'] * total_golden_gate_park_shuttle_ridership['exposure']

In [57]:
total_golden_gate_park_shuttle_ridership.head(5)

,Date,Month,Day Type,Day,Stop,start_date,end_date,Ridership,exposure,Boardings
0,2024-07-01,7,Weekday,Monday,10th Ave/ De Young,2024-07-01,2024-07-01,9,1,9
1,2024-07-01,7,Weekday,Monday,8th Ave,2024-07-01,2024-07-01,7,1,7
2,2024-07-01,7,Weekday,Monday,Academy of Sciences,2024-07-01,2024-07-01,22,1,22
3,2024-07-01,7,Weekday,Monday,Blue Heron Boathouse,2024-07-01,2024-07-01,45,1,45
4,2024-07-01,7,Weekday,Monday,Conservatory of Flowers,2024-07-01,2024-07-01,20,1,20
